# Airbnb Paris – Exp 2: Enhanced Baseline-Modelle
- Gleiche Baselines auf verschiedenen Repräsentationen: cleaned vs. semantisch (PCA/no PCA) vs. enhanced (PCA/no PCA) vs. enhanced+semantisch
- Alignment über `row_id`, gemeinsamer 70/30-Split; feste Detektor-Params (fairer Vergleich)

In [ ]:
import time
import numpy as np
import pandas as pd
import mlflow
from sklearn.model_selection import train_test_split
from sklearn.metrics import average_precision_score, roc_auc_score, precision_recall_curve, auc
from pyod.models.iforest import IForest
from pyod.models.loda import LODA
from pyod.models.ecod import ECOD
from pyod.models.auto_encoder import AutoEncoder

## Repräsentationen laden (indexiert über row_id)
- Outlier = `is_top_rating == 0`; enhanced+semantisch = PCA30-Konkatenation

In [8]:
LABEL = "is_top_rating"
ds = "airbnb_paris"

def_load = lambda name: pd.read_csv(f"../../data/preprocessed/{name}_{ds}.csv").set_index("row_id")
cleaned = def_load("cleaned")
semantic_pca100 = def_load("semantic_pca100")
semantic_pca = def_load("semantic_pca30")
enhanced = def_load("enhanced")
enhanced_pca = def_load("enhanced_pca30")

reps = {
    "cleaned": cleaned.drop(columns=[LABEL]),
    "semantic_pca100": semantic_pca100.drop(columns=[LABEL]),
    "semantic_pca30": semantic_pca.drop(columns=[LABEL]),
    "enhanced": enhanced.drop(columns=[LABEL]),
    "enhanced_pca30": enhanced_pca.drop(columns=[LABEL]),
    "enhanced_semantic_pca30": enhanced_pca.drop(columns=[LABEL]).join(
        semantic_pca.drop(columns=[LABEL]), how="inner", lsuffix="_enh", rsuffix="_sem"),
}

## Gemeinsamer Index & Split
- Schnittmenge aller Repräsentationen (robust gegen unvollständige semantic-CSVs)

In [9]:
common = cleaned.index
for r in reps.values():
    common = common.intersection(r.index)
common = common.sort_values()
y = (1 - cleaned.loc[common, LABEL]).values
print("common rows:", len(common), "outlier rate", round(y.mean(), 4))

tr_id, te_id = train_test_split(common, test_size=0.3, stratify=y, random_state=42)
y_train = (1 - cleaned.loc[tr_id, LABEL]).values
y_test = (1 - cleaned.loc[te_id, LABEL]).values

mlflow.set_tracking_uri("file:../../mlruns")
mlflow.set_experiment("airbnb_paris_experiment_2")

common rows: 18350 outlier rate 0.0393


<Experiment: artifact_location='file:///home/debian/TFM_master_thesis/airbnb_notebooks/exp2/../../mlruns/394247669675461859', creation_time=1781090597525, experiment_id='394247669675461859', last_update_time=1781090597525, lifecycle_stage='active', name='airbnb_paris_experiment_2', tags={}, trace_location=None, workspace='default'>

## Detektoren x Repräsentationen
- Beste Hyperparameter aus Exp 1 (README), **kein GridSearch**; AutoEncoder unsupervised (Originalverteilung, GPU)

In [ ]:
# beste Hyperparameter aus Experiment 1 (README) — kein GridSearch in Exp 2
detectors = {
    "iforest": (IForest, {"n_estimators": 100, "max_features": 1.0, "random_state": 42}, False),
    "loda": (LODA, {"n_bins": 10, "n_random_cuts": 100}, False),
    "ecod": (ECOD, {}, False),
    "autoencoder": (AutoEncoder, {"hidden_neuron_list": [64, 32], "epoch_num": 50, "random_state": 42, "device": "cuda"}, False),
}

for rep_name, rep in reps.items():
    Xtr = rep.loc[tr_id].values
    Xte = rep.loc[te_id].values
    for det_name, (Model, params, inlier_only) in detectors.items():
        t0 = time.perf_counter()
        Xfit = Xtr[y_train == 0] if inlier_only else Xtr
        model = Model(**params)
        model.fit(Xfit)
        scores = model.decision_function(Xte)
        runtime = time.perf_counter() - t0
        ap = average_precision_score(y_test, scores)
        prec, rec, _ = precision_recall_curve(y_test, scores)
        auprc = auc(rec, prec)
        auroc = roc_auc_score(y_test, scores)
        with mlflow.start_run(run_name=f"{rep_name}__{det_name}"):
            mlflow.log_param("representation", rep_name)
            mlflow.log_param("detector", det_name)
            mlflow.log_param("n_features", rep.shape[1])
            mlflow.log_metric("average_precision", ap)
            mlflow.log_metric("auprc", auprc)
            mlflow.log_metric("auc_roc", auroc)
            mlflow.log_metric("runtime_s", runtime)
        print(f"{rep_name:24s} {det_name:12s} AP={ap:.4f} AUPRC={auprc:.4f} AUC={auroc:.4f} feat={rep.shape[1]} t={runtime:.1f}s")